In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "http://old.bdlaws.minlaw.gov.bd/"
INDEX_URL = "http://old.bdlaws.minlaw.gov.bd/bangladesh-code-chronological-index.html?lang=bn"

SAVE_PATH = "/content/drive/MyDrive/bdlaws_links.txt"

headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(INDEX_URL, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

links = set()

for a in soup.find_all("a", href=True):
    href = a["href"].strip()

    if not href or href.startswith("#"):
        continue

    full_url = urljoin(BASE_URL, href)

    if "bdlaws.minlaw.gov.bd" in full_url:
        links.add(full_url)

print(f"Total links collected: {len(links)}")

# Save to Google Drive
with open(SAVE_PATH, "w") as f:
    for link in links:
        f.write(link + "\n")

print(f"Saved to: {SAVE_PATH}")

Total links collected: 1079
Saved to: /content/drive/MyDrive/bdlaws_links.txt


In [ ]:
import os
import requests

# ===== PATHS =====
FILES = [
    "/content/drive/MyDrive/bdlaws_links.txt",
    "/content/drive/MyDrive/bdlaws_links.txt"
]

SAVE_DIR = "/content/drive/MyDrive/bdlaws_pdfs"
os.makedirs(SAVE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})


# ---------- STEP 1: CLEAN ----------
def clean_links(files):
    valid_links = set()

    for file in files:
        with open(file, "r", errors="ignore") as f:
            for line in f:
                url = line.strip()

                # fix missing http
                if url.startswith("bdlaws.minlaw.gov.bd"):
                    url = "http://" + url

                # keep only valid PDFs
                if ".pdf" in url and "upload/bdcodeact" in url:
                    if url.endswith(".pdf"):
                        valid_links.add(url)

    return list(valid_links)


pdf_links = clean_links(FILES)
print(f"Valid PDFs after cleaning: {len(pdf_links)}")


# ---------- STEP 2: DOWNLOAD ----------
def download(url, index, total):
    filename = url.split("/")[-1]
    filepath = os.path.join(SAVE_DIR, filename)

    if os.path.exists(filepath):
        print(f"[{index}/{total}] Skip: {filename}")
        return

    try:
        r = session.get(url, timeout=20)
        if r.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(r.content)
            print(f"[{index}/{total}] Downloaded: {filename}")
        else:
            print(f"[{index}/{total}] Failed ({r.status_code})")

    except Exception as e:
        print(f"[{index}/{total}] Error: {e}")


# ---------- STEP 3: RUN ----------
total = len(pdf_links)

for i, url in enumerate(pdf_links, 1):
    download(url, i, total)

Valid PDFs after cleaning: 1064
[1/1064] Downloaded: 2024-01-29-14-31-19-346.-Law-No.-19-of-2013-(347-356).pdf
[2/1064] Downloaded: 2024-01-22-12-40-24-129.-Act-No.-15-of-2000.pdf
[3/1064] Downloaded: 2023-12-14-11-15-13-379.-THE-IMPORT-OF-GOODS_Final_18.01.2006.pdf
[4/1064] Downloaded: 2023-12-24-11-05-45-4.-Ordinance-No.-XLI-of-1983_fin.pdf
[5/1064] Downloaded: 2023-12-24-09-51-32-27.-Nationalised-Banks-(Transfer-of-Business)-Act,-1975.pdf
[6/1064] Downloaded: 2023-12-24-11-21-06-12.-Income-tax-final.pdf
[7/1064] Downloaded: 2024-01-31-09-14-58-356.-Law-No.-37-of-2013-(547-590).pdf
[8/1064] Downloaded: 2023-12-24-09-39-13-9.-Muslim-Marriages-and-Divorces-(Registration)-Act,-1974.pdf
[9/1064] Downloaded: 2024-02-04-11-04-11-7.-Act-No-24-of-2016.pdf
[10/1064] Downloaded: 2020-11-04-10-44-18-316_THE_EAST_PAKISTAN_HATS_BAZARS_ESTABLISHMENT_A.pdf
[11/1064] Downloaded: 2024-01-22-13-13-52-176.-Act-No-12-of-2002.pdf
[12/1064] Downloaded: 2023-12-03-11-04-31-41.-The-Law-Reports-Act,-1875.pdf